In [1]:


from duckdb_assistant import DuckDBAssistant

dda = DuckDBAssistant()

prompt = """
I found a CSV located in https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv. 
I want to read it, filter out only bad records (i.e. defaulters) and write them
out to a parquet file in my laptop at /Users/ss/duckdb-assistant/example_file.parquet.

"""

In [2]:
response_dict = dda.generate(prompt)

print(response_dict)

{'query': "COPY (\n    SELECT *\n    FROM 'https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv'\n    WHERE BAD = 1\n) TO '/Users/ss/duckdb-assistant/example_file.parquet' (FORMAT PARQUET);", 'response_text': "```sql\nCOPY (\n    SELECT *\n    FROM 'https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv'\n    WHERE BAD = 1\n) TO '/Users/ss/duckdb-assistant/example_file.parquet' (FORMAT PARQUET);\n```"}


In [9]:
from pprint import pprint
pprint(response_dict['query'], indent=4)

('INSTALL httpfs;\n'
 'LOAD httpfs;\n'
 '\n'
 'COPY (\n'
 '    SELECT *\n'
 '    FROM '
 "read_csv_auto('https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv')\n"
 '    WHERE BAD = 1\n'
 ") TO '/Users/ss/duckdb-assistant/example_file.parquet' (FORMAT PARQUET);")


In [4]:
response_dict = dda.generate(prompt, explain_results = True)
from pprint import pprint
pprint(response_dict["response_text"], indent=4)

('```sql\n'
 'COPY (\n'
 '    SELECT *\n'
 '    FROM '
 "read_csv_auto('https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv')\n"
 '    WHERE BAD = 1\n'
 ") TO '/Users/ss/duckdb-assistant/example_file.parquet' (FORMAT PARQUET);\n"
 '```\n'
 '\n'
 '### Explanation\n'
 '\n'
 '1. **Remote CSV Reading (`read_csv_auto`)**: DuckDB uses its `httpfs` '
 'functionality (automatically loaded when accessing `https://` URLs) combined '
 'with `read_csv_auto` to read and automatically infer the schema of the CSV '
 'file directly from GitHub.\n'
 '2. **Filtering Defaulters (`WHERE BAD = 1`)**: In the standard Home Equity '
 '(`hmeq`) dataset, the binary target column `BAD` indicates whether an '
 'applicant defaulted or had severe delinquencies (`1` for bad/defaulter, `0` '
 'for good). We filter the rows to keep only records where `BAD = 1`.\n'
 '3. **Exporting to Parquet (`COPY ... TO ... FORMAT PARQUET`)**: The `COPY` '
 'statement writes the quer

### Summary Explanation

 
 '1. **Dependencies (`httpfs` extension)**: 
 '   To read files directly over HTTP/HTTPS URLs, DuckDB relies on the '
 '`httpfs` extension. Modern versions of DuckDB automatically install and load '
 'extensions on demand, but explicitly running `INSTALL httpfs;` and `LOAD '
 'httpfs;` ensures compatibility across environments where autoloading might '
 'be disabled.

 
 '2. **Reading and Auto-detection**: 
 '   The `read_csv_auto()` function reads the CSV file directly from the '
 'GitHub URL without requiring manual download. It automatically infers column '
 'names and data types (such as interpreting the `BAD` column as an integer).

 
 '3. **Filtering Defaulters**: 
 '   In the standard ______ dataset, the `BAD` column is a flag where `1` '
 'indicates that an applicant defaulted on their loan (a "bad" record) and `0` '
 'indicates repayment. The query filters for `WHERE BAD = 1`.

 '4. **Exporting to Parquet**: 
 '   The `COPY (...) TO ... (FORMAT PARQUET)` statement writes the query '
 'result directly out to a compressed, optimized Parquet file at the specified '
 'local path (`/Users/ss/duckdb-assistant/example_file.parquet`).')

In [5]:
response_dict = dda.generate(prompt, explain_results = True, use_rag=True)
from pprint import pprint
pprint(response_dict["response_text"], indent=4)

('```sql\n'
 'COPY (\n'
 '    SELECT *\n'
 '    FROM '
 "'https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv'\n"
 '    WHERE BAD = 1\n'
 ')\n'
 "TO '/Users/ss/duckdb-assistant/example_file.parquet'\n"
 '(FORMAT PARQUET);\n'
 '```\n'
 '\n'
 '### Summary Explanation\n'
 '\n'
 '1. **Remote CSV Access**: DuckDB uses its built-in HTTP support (`httpfs` '
 'extension, auto-loaded upon demand) to read the CSV directly from the '
 'provided GitHub HTTPS URL without needing to manual download it first.\n'
 '2. **Filtering Defaulters**: In the standard Home Equity (`hmeq`) dataset, '
 'the column `BAD` is a binary flag where `1` represents an applicant who '
 'defaulted on their loan (a bad record). We filter for `WHERE BAD = 1`.\n'
 '3. **Parquet Export**: The `COPY ... TO ... (FORMAT PARQUET)` statement '
 'writes the query output directly into a local Parquet file at '
 '`/Users/ss/duckdb-assistant/example_file.parquet`.')


In [6]:
response_dict = dda.generate(prompt, explain_results = True, use_rag=False)
from pprint import pprint
pprint(response_dict["response_text"], indent=4)

('```sql\n'
 '-- Load the httpfs extension required to read files over HTTP/HTTPS\n'
 'INSTALL httpfs;\n'
 'LOAD httpfs;\n'
 '\n'
 '-- Read the remote CSV, filter for bad records (defaulters where BAD = 1), '
 'and export to Parquet\n'
 'COPY (\n'
 '    SELECT *\n'
 '    FROM '
 "read_csv_auto('https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv')\n"
 '    WHERE BAD = 1\n'
 ") TO '/Users/ss/duckdb-assistant/example_file.parquet' (FORMAT PARQUET);\n"
 '```\n'
 '\n'
 '### Explanation\n'
 '\n'
 '1. **Extension Loading (`httpfs`)**: DuckDB requires the `httpfs` extension '
 'to query remote files over `http://` or `https://` protocol URLs directly.\n'
 '2. **Reading the Remote CSV (`read_csv_auto`)**: DuckDB dynamically streams '
 'and detects the schema, headers, and delimiters of the HMEQ dataset directly '
 'from the GitHub URL without requiring you to download the CSV manually '
 'first.\n'
 '3. **Filtering Defaulters (`WHERE BAD = 1`)*

In [7]:
results = dda.search("I want to read CSV and write to parquet")

In [9]:
dda.sync_docs()

Cloning into '/Users/ss/duckdb-assistant/scripts/chroma'...
Already on 'main'


Your branch is up to date with 'origin/main'.


'417 documents available in collection duckdb_docs'

In [8]:
len(results["documents"])

2

In [9]:
results = dda.search(prompt)

In [10]:
dda.sql(prompt)

In [11]:
dda.execute(prompt)